# Judge SD measurement
Re-scores a stratified subset of the baseline images 5x each to estimate judge noise. The band derived here is the judge component alone; generation noise is the larger term and is not measured here. The end-to-end figure comes from the identical-config rerun, in prompting_check. Reuses the score_baseline.ipynb scoring path.

In [ ]:
# clone OmniGen2, install regex (json_util dep), add to path
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/VectorSpaceLab/OmniGen2.git", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex"], check=True)
sys.path.insert(0, REPO)
os.chdir(REPO)
print("ready")

ready


In [ ]:
# patch evaluate(): break after a successful parse (2 calls/row, not 6)
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


In [ ]:
# mount Drive, OpenAI key, paths; restore reference crops from tar if absent
from google.colab import drive, userdata
import os, subprocess
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
GEN   = f"{DRIVE}/gen/baseline"
IMGS, EXPECTED = "/content/imgs", 37063
TAR   = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0
if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
print("reference images:", n_files(IMGS))
if not os.path.exists("/content/test.jsonl"):
    subprocess.run(["cp", f"{DRIVE}/test.jsonl", "/content/test.jsonl"])
print("baseline images:", len([f for f in os.listdir(GEN) if f.endswith(".png")]) if os.path.isdir(GEN) else "MISSING")

Mounted at /content/drive
reference images: 37063
baseline images: 488


In [ ]:
# stratified subset: PER rows per reference count, fixed seed
import json, random
rows = [json.loads(l) for l in open("/content/test.jsonl", encoding="utf-8")]
PER, REPS, SEED = 30, 5, 42

by_ref = {1: [], 2: [], 3: []}
for i, r in enumerate(rows):
    by_ref[len(r["input_images"])].append(i)
rng = random.Random(SEED)
SUBSET = sorted(j for k in (1, 2, 3) for j in rng.sample(by_ref[k], min(PER, len(by_ref[k]))))
print("subset rows:", len(SUBSET), "| per ref-count:", {k: min(PER, len(by_ref[k])) for k in (1, 2, 3)}, "| reps:", REPS)

subset rows: 90 | per ref-count: {1: 30, 2: 30, 3: 30} | reps: 5


In [ ]:
# score each subset row REPS times; resumable to sd_reps.jsonl
import os, json, math, time
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))
REPS_PATH = f"{DRIVE}/scores/sd_reps.jsonl"
os.makedirs(os.path.dirname(REPS_PATH), exist_ok=True)

def resize_ref(im):
    w, h = im.size; cap = 1024 * 1024
    if w * h > cap:
        r = math.sqrt(cap / (w * h)); w, h = int(w * r), int(h * r)
    return im.resize((max(16, w // 16 * 16), max(16, h // 16 * 16)))

def num(x):
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

done = set()
if os.path.exists(REPS_PATH):
    for l in open(REPS_PATH, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add((d["i"], d["rep"]))
        except: pass

def score_one(i, rep):
    r = rows[i]; gp = f"{GEN}/test_{i:05d}.png"
    if not os.path.exists(gp): return {"i": i, "rep": rep, "error": "missing_generated"}
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen = Image.open(gp).convert("RGB")
    time.sleep(0.2)
    res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
    if pf is None or sc is None: return {"i": i, "rep": rep, "error": "unparsable_score"}
    return {"i": i, "rep": rep, "n_refs": len(r["input_images"]), "PF": pf, "SC": sc}

tasks = [(i, rep) for i in SUBSET for rep in range(REPS) if (i, rep) not in done]
print(f"to score: {len(tasks)} (done: {len(done)})")
with open(REPS_PATH, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(score_one, i, rep) for i, rep in tasks]
    for n, fut in enumerate(as_completed(futs), 1):
        f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
        if n % 50 == 0: print(f"  {n}/{len(tasks)}")
print("done ->", REPS_PATH)

to score: 0 (done: 450)
done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/sd_reps.jsonl


In [ ]:
# per-item SD -> test-weighted sigma_item -> sigma_mean -> judge-only tie band
import json, math
from collections import defaultdict
REPS_PATH = f"{DRIVE}/scores/sd_reps.jsonl"

recs, errs = [], defaultdict(int)
for l in open(REPS_PATH, encoding="utf-8"):
    d = json.loads(l)
    if "error" in d: errs[d["error"]] += 1; continue
    recs.append(d)

byrow = defaultdict(list)
for d in recs: byrow[d["i"]].append(d)

row_stats = {}
for i, ds in byrow.items():
    ov = [math.sqrt(d["PF"] * d["SC"]) for d in ds]
    if len(ov) < 2: continue
    m = sum(ov) / len(ov)
    row_stats[i] = {"n_refs": ds[0]["n_refs"], "mean": m,
                    "ss": sum((x - m) ** 2 for x in ov), "df": len(ov) - 1}

def pooled(items):
    ss = sum(s["ss"] for s in items); df = sum(s["df"] for s in items)
    return (math.sqrt(ss / df) if df else float("nan")), df, len(items)

print("reps collected:", len(recs), "| errors:", dict(errs), "| rows with >=2 reps:", len(row_stats))

print("\nper-item Overall SD by reference count:")
sigma_s = {}
for k in (1, 2, 3):
    its = [s for s in row_stats.values() if s["n_refs"] == k]
    if its:
        sd, df, nr = pooled(its); sigma_s[k] = sd
        print("  %d-ref  rows=%d  sigma_item=%.3f  (df=%d)" % (k, nr, sd, df))

testn = defaultdict(int)
for r in rows: testn[len(r["input_images"])] += 1
NT = sum(testn.values())
sigma_item = math.sqrt(sum((testn[k] / NT) * sigma_s[k] ** 2 for k in sigma_s))
sigma_mean = sigma_item / math.sqrt(NT)
tie = 2.8 * sigma_mean
print("\nweighted to test mix", dict(testn), "n=%d" % NT)
print("  sigma_item = %.3f" % sigma_item)
print("  sigma_mean (SD of full-test Overall mean) = sigma_item/sqrt(%d) = %.4f" % (NT, sigma_mean))
print("  judge-only tie band = 2.8*sigma_mean = %.3f Overall" % tie)
print("  covers judge re-scoring of a fixed image set. Generation noise is not in this figure and")
print("  is the larger term; the end-to-end band is measured by the identical-config rerun.")

print("\nper-item Overall SD by score level:")
for lo, hi in [(0, 6), (6, 7.5), (7.5, 9), (9, 10.01)]:
    its = [s for s in row_stats.values() if lo <= s["mean"] < hi]
    if its:
        sd, df, nr = pooled(its)
        print("  [%.1f,%.1f)  rows=%d  sigma_item=%.3f" % (lo, hi, nr, sd))

BP = f"{DRIVE}/scores/test__baseline.jsonl"
if os.path.exists(BP):
    base = {}
    for l in open(BP, encoding="utf-8"):
        d = json.loads(l)
        if "error" not in d: base[d["i"]] = math.sqrt(d["PF"] * d["SC"])
    common = [i for i in row_stats if i in base]
    if common:
        fresh = sum(row_stats[i]["mean"] for i in common) / len(common)
        old = sum(base[i] for i in common) / len(common)
        print("\ndrift check (subset n=%d): fresh Overall mean=%.3f  baseline=%.3f  diff=%.3f" % (len(common), fresh, old, fresh - old))

reps collected: 450 | errors: {} | rows with >=2 reps: 90

per-item Overall SD by reference count:
  1-ref  rows=30  sigma_item=0.393  (df=120)
  2-ref  rows=30  sigma_item=0.292  (df=120)
  3-ref  rows=30  sigma_item=0.393  (df=120)

weighted to test mix {1: 243, 2: 194, 3: 51} n=488
  sigma_item = 0.356
  sigma_mean (SD of full-test Overall mean) = sigma_item/sqrt(488) = 0.0161
  judge-only tie band = 2.8*sigma_mean = 0.045 Overall
  covers judge re-scoring of a fixed image set. Generation noise is not in this figure and
  is the larger term; the end-to-end band is measured by the identical-config rerun.

per-item Overall SD by score level:
  [0.0,6.0)  rows=6  sigma_item=0.686
  [6.0,7.5)  rows=14  sigma_item=0.538
  [7.5,9.0)  rows=50  sigma_item=0.304
  [9.0,10.0)  rows=20  sigma_item=0.129

drift check (subset n=90): fresh Overall mean=8.098  baseline=7.986  diff=0.112
